# The eCREAM federation in ten minutes: the live demo as a notebook

_The five screens of the web-interface demo and the optional third experiment, as federated
calls with the `mip` client. Both synthetic datasets, 5,000 subjects each; no real patient data.
Detail, expected results and interpretation: `docs/teaching_scenarios/` and the three analysis
notebooks next to this one._

**Who is seen by a doctor sooner: dyspnoea or TLoC?** Say the question out loud, then let the
platform answer it. What to watch for: no file opens, no row is shown, every number is a
statistic computed where the data live.

## 1. Choose the data

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("UC1")         # add version="eCREAMv8" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the federation labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

dysp = dataset("DYSP5K", "Synthetic Dyspnea dataset (5000 subjects)")
tloc = dataset("TLOC5K", "Synthetic TLoC dataset (5000 subjects)")
DATASETS = [dysp, tloc]                   # every call of this notebook runs on both synthetic datasets
display(dm)
display(dm.datasets.to_frame())           # 21 dataset codes: four hold data today, 17 are the EDs' containers

In [ ]:
delay   = dm.variables["Door to doctor"]
sex     = dm.variables["Sex"]
age     = dm.variables["Age"]
dataset = dm.variables["Dataset Registry origin"]        # which cohort a record belongs to

from mip.preprocessing import MissingValuesHandler
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time

def analysis(variables, filters=None, drop_missing=False, datasets=None):
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)

print("10,000 records: both synthetic datasets, four variables")

## 2. Review before modelling

One federated `describe()`: one row per dataset and one pooled row, no filter.
*Expected:* door to doctor 47.8 minutes on average with dyspnoea (median 41), 105.3 with TLoC
(median 53); no missing value.

In [ ]:
t0 = time.perf_counter()
review = analysis([delay, sex, age]).describe()
print(f"answered in {time.perf_counter() - t0:.1f} s")
frame = review.to_frame()                                    # n, mean, SD, min, max per dataset and pooled
display(frame[frame["variable"].isin([delay._code, age._code])])
medians = {r["dataset"]: r["data"]["q2"] for r in review.raw["featurewise"] if r["variable"] == delay._code and r["data"]}
print("median door to doctor:", medians)

## 3. Compare the cohorts: Independent t-test

*Expected:* a difference of means near -57 minutes (dyspnoea minus TLoC), t = -34, p far below
0.001.

In [ ]:
t0 = time.perf_counter()
t = analysis([delay, dataset], drop_missing=True).t_test(variable=delay, group_by=dataset, group_a="DYSP5K", group_b="TLOC5K")
raw = t.raw
print(f"answered in {time.perf_counter() - t0:.1f} s")
print(f"difference of means {raw['mean_diff']:.1f} minutes (95% CI {raw['ci_lower']:.1f} to {raw['ci_upper']:.1f}), "
      f"t = {raw['t_stat']:.1f}, p = {raw['p']:.1e}, Cohen's d {raw['cohens_d']:.2f}")

## 4. A categorical look: Sex by cohort, Chi-squared Test

*Expected:* 68% men with dyspnoea against 48% with TLoC; chi-square about 390, p of the order of
10^-87 (with the "Unknown" sex code filtered out).

In [ ]:
levels = {"1": "Male", "2": "Female", "3": "Unknown"}
counts = pd.DataFrame({r["dataset"]: {levels.get(k, k): v for k, v in r["data"]["counts"].items()}
                       for r in review.raw["featurewise"] if r["variable"] == sex._code and r["data"]})
display(counts)
display((100 * counts / counts.sum()).round(1))
chi = analysis([sex, dataset], F(sex).isin(["1", "2"]), drop_missing=True).chi_square_test(x=dataset, y=sex)
print(f"chi-square {chi.raw['chi2']:.1f} on {chi.raw['dof']} df, p = {chi.raw['p_value']:.1e}")

## 5. Save the experiment

In the web interface the demo ends with **Save experiment**: a named, re-openable, shareable
object in My experiments. The notebook can list what was saved.

In [ ]:
saved = client.experiments().list()
display(pd.DataFrame(saved) if saved else "No saved experiment is listed yet.")

## Optional third experiment: a model that must find nothing

30-day mortality on age, sex and oxygen saturation, with cross-validation. *Expected:* an area
under the ROC curve near 0.5, because the synthetic generator resampled the demographic,
vital-sign and follow-up blocks independently: the platform reports no signal because there is
none. (The filter keeps the known outcomes and the known sexes: without it the 29 records of
unknown sex, which hold no death, make the model unfittable.)

In [ ]:
mortality = dm.variables["30-day mortality"]
spo2 = dm.variables["SpO2 room air (1st)"]
cohort = F(mortality).isin(["1", "2"]) & F(sex).isin(["1", "2"])
t0 = time.perf_counter()
cv = analysis([mortality, age, sex, spo2], cohort, drop_missing=True).logistic_regression_cv(x=[age, sex, spo2], y=mortality, positive_class="1", n_splits=3)
print(f"answered in {time.perf_counter() - t0:.1f} s")
curves, cm = cv.raw["roc_curves"], cv.raw["confusion_matrix"]
print("area under the ROC curve per fold:", ", ".join(f"{c['auc']:.2f}" for c in curves), f"| average {np.mean([c['auc'] for c in curves]):.2f}")
print(f"confusion matrix at the 50% cut-off: {cm}; accuracy {100 * (cm['tp'] + cm['tn']) / sum(cm.values()):.1f}%, "
      f"deaths caught {cm['tp']} of {cm['tp'] + cm['fn']}")
fig, ax = plt.subplots(figsize=(4.5, 4))
for c in curves:
    ax.plot(c["fpr"], c["tpr"], linewidth=1, label=f"{c['name'].replace('_', ' ')}: area {c['auc']:.2f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss")
ax.set_xlabel("survivors wrongly called deaths"); ax.set_ylabel("deaths caught"); ax.legend(fontsize=8)
ax.set_title("A negative control: no signal to find")
plt.tight_layout()
plt.show()

**Synthetic data, real workflow.** The contrast between the cohorts is plausible (visible
breathlessness is seen faster than a collapse that has resolved) but is not a finding; the
mortality model finds nothing because nothing is there. The point is that the platform runs end
to end on the eCREAM data model, from the notebook as from the web interface, with statistics
and never data crossing the line.